# S45_05 — Accelerate and Distributed Training

HuggingFace **Accelerate** abstracts hardware differences so the same training code runs on a laptop CPU, a single GPU, or a multi-GPU / multi-node cluster without code changes.

## The Accelerate abstraction

In [ ]:
# pip install accelerate
# Configure hardware once (interactive CLI, or via config file):
# accelerate config

from accelerate import Accelerator
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

accelerator = Accelerator(
    mixed_precision='fp16',   # 'fp16' | 'bf16' | 'no'
    # gradient_accumulation_steps=4,   # simulate larger batch size
)

print(f'Device: {accelerator.device}')
print(f'Mixed precision: {accelerator.mixed_precision}')
print(f'Num processes: {accelerator.num_processes}')

In [ ]:
# Minimal training loop — identical code runs on any hardware
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Toy dataset
X = torch.randint(0, 1000, (100, 32))    # 100 sequences, length 32
masks = torch.ones(100, 32, dtype=torch.long)
y = torch.randint(0, 2, (100,))

dataset = TensorDataset(X, masks, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# accelerator.prepare() wraps everything — handles device placement, DDP, FP16
model, optimizer, loader = accelerator.prepare(model, optimizer, loader)

model.train()
for epoch in range(1):
    for input_ids, attention_mask, labels in loader:
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        accelerator.backward(loss)   # NOT loss.backward() — Accelerate handles scaling
        optimizer.step()

print(f'Training complete. Final loss: {loss.item():.4f}')

## Mixed precision — FP16 and BF16

In [ ]:
import torch

# FP32 vs FP16 memory comparison
param_count = 110_000_000  # BERT-base

fp32_mb = param_count * 4 / 1e6
fp16_mb = param_count * 2 / 1e6

print(f'BERT-base parameters: {param_count:,}')
print(f'FP32 weight size: {fp32_mb:.0f} MB')
print(f'FP16 weight size: {fp16_mb:.0f} MB (2× smaller)')
print()

# PyTorch autocast — manual mixed precision without Accelerate
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Pattern: autocast for forward pass, GradScaler to prevent underflow
print('''
Manual mixed precision pattern:

    scaler = torch.cuda.amp.GradScaler()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = model(input)
        loss = criterion(output, target)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

Accelerate handles this automatically with mixed_precision="fp16".
''')

## Gradient accumulation — simulate larger batches

In [ ]:
# Gradient accumulation: accumulate gradients over N steps, then update
# Effective batch size = per_device_batch * n_gpus * accumulation_steps

accumulation_steps = 4
accelerator_accum = Accelerator(
    gradient_accumulation_steps=accumulation_steps,
    mixed_precision='fp16',
)

# In training loop:
print('''
With gradient accumulation:

    for step, batch in enumerate(loader):
        with accelerator.accumulate(model):   # handles zero_grad and step timing
            outputs = model(**batch)
            accelerator.backward(outputs.loss)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

accelerate.accumulate() is the clean way — no manual step counting.
''')

## Launching distributed training

In [ ]:
# To run on multiple GPUs, launch with:
# accelerate launch --num_processes=4 train.py

# Or configure via accelerate config and then:
# accelerate launch train.py

# The Trainer from transformers uses Accelerate internally — same benefits
# without writing your own loop

print('Distributed training options (all use Accelerate internally):')
print('  Single GPU:      python train.py')
print('  Multi-GPU:       accelerate launch --num_processes=4 train.py')
print('  Multi-node:      accelerate launch --multi_gpu --machine_rank=... train.py')
print('  TPU (Colab/GCP): accelerate launch --tpu train.py')
print()
print('The Trainer API handles this automatically with TrainingArguments.')

## Memory-efficient training summary

| Technique | Memory saving | When to use |
|-----------|--------------|-------------|
| FP16/BF16 | ~2× | Always on modern GPU |
| Gradient accumulation | Reduces peak batch memory | Small GPU, large model |
| Gradient checkpointing | Up to 10× (at 30% compute cost) | Large model, memory-limited |
| LoRA/PEFT | Reduces trainable params | Few-shot fine-tuning |
| 4-bit (QLoRA) | ~4× vs FP16 | Consumer GPU (<24GB VRAM) |
| DeepSpeed ZeRO | Shards across GPUs | Multi-GPU, large models |

**Gradient checkpointing** (trading compute for memory): `model.gradient_checkpointing_enable()`

This completes S45. Next section: [S46_LLMs_Prompting_Evaluation](../03_LLMs_Prompting_Evaluation/S46_01_what_are_llms.ipynb)